
# Préparation des données pour la modélisation RH

Objectif : produire deux objets directement exploitables par scikit-learn :

- `X` : DataFrame de features numériques, sans valeur manquante, sans colonne cible, sans identifiant technique ;
- `y` : Series cible encodée en `0/1`, où `1` signifie que l'employé a quitté l'entreprise.

Ce notebook reprend les fichiers sources, reconstruit le DataFrame central, puis applique les transformations nécessaires pour obtenir `X` et `y`.


In [1]:

import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report


## 1. Chemins des fichiers

In [3]:

DATA_DIR = Path('.')

SIRH_PATH = DATA_DIR / 'extrait_sirh.csv'
EVAL_PATH = DATA_DIR / 'extrait_eval.csv'
SONDAGE_PATH = DATA_DIR / 'extrait_sondage.csv'

# Si vous travaillez dans un autre dossier, adaptez DATA_DIR ou les chemins ci-dessus.



## 2. Fonctions de chargement et de jointure

On transforme les clés de jointure pour obtenir une colonne commune `id_employee` dans les trois fichiers.


In [4]:

def charger_fichiers(sirh_path: Path, eval_path: Path, sondage_path: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Charge les trois fichiers sources RH."""
    df_sirh = pd.read_csv(sirh_path)
    df_eval = pd.read_csv(eval_path)
    df_sondage = pd.read_csv(sondage_path)
    return df_sirh, df_eval, df_sondage


def construire_dataframe_central(
    df_sirh: pd.DataFrame,
    df_eval: pd.DataFrame,
    df_sondage: pd.DataFrame,
) -> pd.DataFrame:
    """Construit le DataFrame central par jointure one-to-one entre les trois sources."""
    df_eval = df_eval.copy()
    df_sondage = df_sondage.copy()

    # eval_number contient des valeurs de type E_1, E_2, etc.
    df_eval["id_employee"] = (
        df_eval["eval_number"]
        .astype(str)
        .str.extract(r"(\d+)")
        .astype(int)
    )

    # code_sondage correspond à l'identifiant employé.
    df_sondage = df_sondage.rename(columns={"code_sondage": "id_employee"})

    # Conversion du pourcentage texte en numérique.
    df_eval["augmentation_salaire_precedente_pct"] = (
        df_eval["augementation_salaire_precedente"]
        .astype(str)
        .str.replace("%", "", regex=False)
        .str.strip()
        .astype(float)
    )

    df_central = (
        df_sirh
        .merge(df_eval, on="id_employee", how="inner", validate="one_to_one")
        .merge(df_sondage, on="id_employee", how="inner", validate="one_to_one")
    )

    return df_central


In [5]:

df_sirh, df_eval, df_sondage = charger_fichiers(SIRH_PATH, EVAL_PATH, SONDAGE_PATH)
df_central = construire_dataframe_central(df_sirh, df_eval, df_sondage)

print(df_central.shape)
df_central.head()


(1470, 34)


,id_employee,age,genre,revenu_mensuel,statut_marital,departement,poste,nombre_experiences_precedentes,nombre_heures_travailless,annee_experience_totale,...,nombre_participation_pee,nb_formations_suivies,nombre_employee_sous_responsabilite,distance_domicile_travail,niveau_education,domaine_etude,ayant_enfants,frequence_deplacement,annees_depuis_la_derniere_promotion,annes_sous_responsable_actuel
0,1,41,F,5993,Célibataire,Commercial,Cadre Commercial,8,80,8,...,0,0,1,1,2,Infra & Cloud,Y,Occasionnel,0,5
1,2,49,M,5130,Marié(e),Consulting,Assistant de Direction,1,80,10,...,1,3,1,8,1,Infra & Cloud,Y,Frequent,1,7
2,4,37,M,2090,Célibataire,Consulting,Consultant,6,80,7,...,0,3,1,2,2,Autre,Y,Occasionnel,0,0
3,5,33,F,2909,Marié(e),Consulting,Assistant de Direction,1,80,8,...,0,3,1,3,4,Infra & Cloud,Y,Frequent,3,0
4,7,27,M,3468,Marié(e),Consulting,Consultant,9,80,6,...,1,3,1,2,1,Transformation Digitale,Y,Occasionnel,2,2



## 3. Fonctions de feature engineering et de préparation `X`, `y`

Principes appliqués :

- la cible `a_quitte_l_entreprise` est transformée en `0/1` ;
- les identifiants techniques sont supprimés ;
- la colonne brute `augementation_salaire_precedente`, au format texte, est remplacée par sa version numérique ;
- les variables catégorielles sont encodées avec `pd.get_dummies()` ;
- les valeurs manquantes sont imputées avant encodage ;
- les colonnes constantes sont supprimées, car elles n'apportent aucune information au modèle.


In [6]:

def ajouter_features_metier(df: pd.DataFrame) -> pd.DataFrame:
    """Ajoute des variables dérivées simples à partir du DataFrame central.

    Ces features ne doivent utiliser que les colonnes explicatives, jamais la cible.
    """
    df = df.copy()

    df["ratio_anciennete_experience"] = np.where(
        df["annee_experience_totale"] > 0,
        df["annees_dans_l_entreprise"] / df["annee_experience_totale"],
        0,
    )

    df["ratio_temps_poste_anciennete"] = np.where(
        df["annees_dans_l_entreprise"] > 0,
        df["annees_dans_le_poste_actuel"] / df["annees_dans_l_entreprise"],
        0,
    )

    df["ratio_promotion_anciennete"] = np.where(
        df["annees_dans_l_entreprise"] > 0,
        df["annees_depuis_la_derniere_promotion"] / df["annees_dans_l_entreprise"],
        0,
    )

    df["revenu_par_annee_experience"] = df["revenu_mensuel"] / (df["annee_experience_totale"] + 1)

    return df


def preparer_X_y(
    df_central: pd.DataFrame,
    target_col: str = "a_quitte_l_entreprise",
    ajouter_features: bool = True,
    supprimer_colonnes_constantes: bool = True,
) -> tuple[pd.DataFrame, pd.Series]:
    """Prépare un DataFrame X et une Series y compatibles avec sklearn.fit()."""
    df = df_central.copy()

    if ajouter_features:
        df = ajouter_features_metier(df)

    # Séparation et encodage de la cible.
    y = (
        df[target_col]
        .map({"Non": 0, "Oui": 1})
        .astype("int64")
        .rename(target_col)
    )

    # Colonnes à exclure des features.
    colonnes_a_exclure = [
        target_col,
        "id_employee",                         # identifiant technique
        "eval_number",                         # identifiant technique redondant
        "augementation_salaire_precedente",    # version texte remplacée par augmentation_salaire_precedente_pct
    ]

    X_raw = df.drop(columns=[c for c in colonnes_a_exclure if c in df.columns])

    # Identification des colonnes numériques et catégorielles.
    colonnes_numeriques = X_raw.select_dtypes(include=["number"]).columns.tolist()
    colonnes_categorielles = X_raw.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

    # Imputation simple des valeurs manquantes.
    for col in colonnes_numeriques:
        X_raw[col] = X_raw[col].fillna(X_raw[col].median())

    for col in colonnes_categorielles:
        mode = X_raw[col].mode(dropna=True)
        valeur_remplacement = mode.iloc[0] if len(mode) > 0 else "Inconnu"
        X_raw[col] = X_raw[col].fillna(valeur_remplacement)

    # Encodage one-hot des variables catégorielles.
    X = pd.get_dummies(
        X_raw,
        columns=colonnes_categorielles,
        drop_first=False,
        dtype=int,
    )

    # Suppression éventuelle des colonnes constantes.
    if supprimer_colonnes_constantes:
        colonnes_constantes = X.columns[X.nunique(dropna=False) <= 1]
        X = X.drop(columns=colonnes_constantes)

    # Tri des colonnes pour assurer une sortie stable et reproductible.
    X = X.sort_index(axis=1)

    return X, y


In [7]:

X, y = preparer_X_y(df_central)

print("Shape de X :", X.shape)
print("Shape de y :", y.shape)
print("Répartition de la cible :")
print(y.value_counts(normalize=True).rename("proportion"))

X.head()


Shape de X : (1470, 52)
Shape de y : (1470,)
Répartition de la cible :
a_quitte_l_entreprise
0    0.838776
1    0.161224
Name: proportion, dtype: float64


C:\Users\johnd\AppData\Local\Temp\ipykernel_6220\3131522134.py:63: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colonnes_categorielles = X_raw.select_dtypes(include=["object", "category", "bool"]).columns.tolist()


,age,annee_experience_totale,annees_dans_l_entreprise,annees_dans_le_poste_actuel,annees_depuis_la_derniere_promotion,annes_sous_responsable_actuel,augmentation_salaire_precedente_pct,departement_Commercial,departement_Consulting,departement_Ressources Humaines,...,ratio_temps_poste_anciennete,revenu_mensuel,revenu_par_annee_experience,satisfaction_employee_environnement,satisfaction_employee_equilibre_pro_perso,satisfaction_employee_equipe,satisfaction_employee_nature_travail,statut_marital_Célibataire,statut_marital_Divorcé(e),statut_marital_Marié(e)
0,41,8,6,4,0,5,11.0,1,0,0,...,0.666667,5993,665.888889,2,1,1,4,1,0,0
1,49,10,10,7,1,7,23.0,0,1,0,...,0.700000,5130,466.363636,3,3,4,2,0,0,1
2,37,7,0,0,0,0,15.0,0,1,0,...,0.000000,2090,261.250000,4,3,2,3,1,0,0
3,33,8,8,7,3,0,11.0,0,1,0,...,0.875000,2909,323.222222,4,3,3,3,0,0,1
4,27,6,2,2,2,2,12.0,0,1,0,...,1.000000,3468,495.428571,1,3,4,2,0,0,1



## 4. Vérification de compatibilité avec scikit-learn

La cellule suivante vérifie que `X` peut être injecté dans une méthode `fit()` sans erreur de type, de valeur manquante ou de variable catégorielle non encodée.


In [8]:

def verifier_compatibilite_sklearn(X: pd.DataFrame, y: pd.Series) -> None:
    """Contrôle que X et y sont utilisables dans sklearn.fit()."""
    assert isinstance(X, pd.DataFrame), "X doit être un DataFrame pandas."
    assert isinstance(y, pd.Series), "y doit être une Series pandas."
    assert len(X) == len(y), "X et y doivent avoir le même nombre de lignes."
    assert X.isna().sum().sum() == 0, "X contient des valeurs manquantes."
    assert y.isna().sum() == 0, "y contient des valeurs manquantes."
    assert len(X.select_dtypes(exclude=["number"]).columns) == 0, "X contient encore des colonnes non numériques."

    # Test réel avec un modèle sklearn peu sensible au scaling.
    modele_test = RandomForestClassifier(n_estimators=50, random_state=42)
    modele_test.fit(X, y)

    print("Validation OK : X et y sont compatibles avec sklearn.fit().")


verifier_compatibilite_sklearn(X, y)


Validation OK : X et y sont compatibles avec sklearn.fit().



## 5. Exemple minimal de modélisation

Ce bloc n'est pas l'objectif principal de l'étape, mais il confirme que les données préparées sont utilisables dans un workflow ML standard.


In [9]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

modele = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
)

modele.fit(X_train, y_train)
y_pred = modele.predict(X_test)
y_proba = modele.predict_proba(X_test)[:, 1]

print("Accuracy :", round(accuracy_score(y_test, y_pred), 3))
print("ROC AUC  :", round(roc_auc_score(y_test, y_proba), 3))
print()
print(classification_report(y_test, y_pred, target_names=["Non", "Oui"]))


Accuracy : 0.84
ROC AUC  : 0.775

              precision    recall  f1-score   support

         Non       0.85      0.98      0.91       247
         Oui       0.50      0.09      0.15        47

    accuracy                           0.84       294
   macro avg       0.67      0.53      0.53       294
weighted avg       0.79      0.84      0.79       294




## 6. Export des objets de modélisation

Ces fichiers peuvent être réutilisés dans un autre notebook ou dans un script de modélisation.


In [10]:

X.to_csv("X_features_modelisation.csv", index=False)
y.to_csv("y_cible_modelisation.csv", index=False)

df_central.to_csv("df_central_rh.csv", index=False)

print("Fichiers exportés :")
print("- X_features_modelisation.csv")
print("- y_cible_modelisation.csv")
print("- df_central_rh.csv")


Fichiers exportés :
- X_features_modelisation.csv
- y_cible_modelisation.csv
- df_central_rh.csv
